<a href="https://colab.research.google.com/github/busybey11/clinical-data-portfolio/blob/main/Project_1_Clinical_Data_Quality_Control_Pipeline_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Project 1: Clinical Data Quality Control Pipeline in Python

This notebook builds a quality control pipeline that checks real clinical trial data for the kinds of issues a Clinical Data Manager looks for during a study. The data used here comes from CDISC's publicly released Pilot Project dataset, a real clinical trial dataset with patient identities removed that is distributed for training purposes.

Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Install pandas

In [2]:
!pip install pandas pyreadstat -q

import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 71.5 MB/s eta 0:00:00


**Fetch CDISC Clinical Trial data (Pilot Project Data in GitHub)**

This step pulls three real domains directly from CDISC's public GitHub repository. The demographics domain, known as DM, contains one row per subject. The vital signs domain, known as VS, contains repeated measurements taken throughout the trial. The adverse events domain, known as AE, contains every adverse event reported during the study. After loading, the character columns are decoded from raw bytes into readable text, since SAS transport files store text as bytes by default.

In [3]:
# CDISC's official Pilot Project clinical trial data
base_url = "https://raw.githubusercontent.com/cdisc-org/sdtm-adam-pilot-project/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm/"

dm = pd.read_sas(base_url + "dm.xpt", format="xport")  # Demographics domain
vs = pd.read_sas(base_url + "vs.xpt", format="xport")  # Vital Signs domain
ae = pd.read_sas(base_url + "ae.xpt", format="xport")  # Adverse Events domain

print(dm.shape, vs.shape, ae.shape)

(306, 25) (29643, 24) (1191, 35)


Fix the byte-string format

In [4]:
for df in [dm, vs, ae]:
    for col in df.select_dtypes([object]).columns:
        df[col] = df[col].str.decode("utf-8").str.strip()

**Check for missing required fields**

This check looks across every record in the demographics domain and flags any subject missing an identifier, an age, or a sex on file.

In [20]:
def check_missing(df, required_cols, domain_name):
    issues = []
    for col in required_cols:
        missing = df[df[col].isna()]
        for idx in missing.index:
            issues.append({
                "domain": domain_name,
                "subject": missing.loc[idx].get("USUBJID", "unknown"),
                "category": "missing_required_field",
                "issue": f"Missing required field: {col}"
            })
    return issues

dm_issues = check_missing(dm, ["USUBJID", "AGE", "SEX"], "DM")
print(f"Found {len(dm_issues)} dm_issues")
dm_issues

Found 0 dm_issues


[]

**Check if there are out of range vital signs**

This check looks at blood pressure and pulse readings in the vital signs domain and flags any reading outside a plausible physiological range. These ranges are illustrative for this project. In an actual study, the thresholds would come from that study's own data validation plan.

In [22]:
# Updated vital sign check
def check_vitals_range(vs_df):
    issues = []
    ranges = {"SYSBP": (70, 200), "DIABP": (40, 120), "PULSE": (40, 150)}
    for _, row in vs_df.iterrows():
        test = row.get("VSTESTCD")
        val = row.get("VSSTRESN")
        if test in ranges and pd.notna(val):
            low, high = ranges[test]
            if not (low <= val <= high):
                issues.append({
                    "domain": "VS",
                    "subject": row.get("USUBJID"),
                    "category": "out_of_range_value",
                    "issue": f"{test} value {val} outside expected range ({low}-{high})"
                })
    return issues

vs_issues = check_vitals_range(vs)
print(f"Found {len(vs_issues)} vs_issues")
vs_issues

Found 7 vs_issues


[{'domain': 'VS',
  'subject': '01-701-1203',
  'category': 'out_of_range_value',
  'issue': 'DIABP value 39.0 outside expected range (40-120)'},
 {'domain': 'VS',
  'subject': '01-701-1203',
  'category': 'out_of_range_value',
  'issue': 'DIABP value 39.0 outside expected range (40-120)'},
 {'domain': 'VS',
  'subject': '01-701-1345',
  'category': 'out_of_range_value',
  'issue': 'DIABP value 39.0 outside expected range (40-120)'},
 {'domain': 'VS',
  'subject': '01-706-1384',
  'category': 'out_of_range_value',
  'issue': 'SYSBP value 217.0 outside expected range (70-200)'},
 {'domain': 'VS',
  'subject': '01-708-1158',
  'category': 'out_of_range_value',
  'issue': 'SYSBP value 208.0 outside expected range (70-200)'},
 {'domain': 'VS',
  'subject': '01-716-1026',
  'category': 'out_of_range_value',
  'issue': 'SYSBP value 210.0 outside expected range (70-200)'},
 {'domain': 'VS',
  'subject': '01-718-1355',
  'category': 'out_of_range_value',
  'issue': 'SYSBP value 202.0 outside e

**Check if there are duplicated subject IDs**

This check looks for any subject identifier that appears more than once in the demographics domain, since every subject should have exactly one demographic record.

In [13]:
def check_duplicates(df, domain_name):
    dupes = df[df.duplicated(subset=["USUBJID"], keep=False)]
    return [{
        "domain": domain_name,
        "subject": s,
        "category": "duplicate_subject_id",
        "issue": "Duplicate USUBJID"
    } for s in dupes["USUBJID"].unique()]

dm_dupes = check_duplicates(dm, "DM")
print(f"Found {len(dm_dupes)} duplicate subject ID issues")
dm_dupes

Found 0 duplicate subject ID issues


[]

**Check for Adverse Event Dates**

This check compares each adverse event's start date against the subject's reference start date. An early date could mean the event was collected during the screening period, before treatment began, which is normal. Or it could mean a previously existing condition was logged with only an approximate date of onset. This check separates those two situations by looking at how precise each date actually is.

In [10]:
from datetime import datetime

def parse_partial_date(date_str):
    """
    Parses an ISO 8601 date that may be full (YYYY-MM-DD),
    month-level (YYYY-MM), or year-level (YYYY) precision.
    Returns (parsed_datetime, precision_label).
    """
    if date_str is None or pd.isna(date_str) or str(date_str).strip() == "":
        return None, "missing"

    date_str = str(date_str).strip()
    try:
        if len(date_str) == 10:
            return datetime.strptime(date_str, "%Y-%m-%d"), "day"
        elif len(date_str) == 7:
            return datetime.strptime(date_str, "%Y-%m"), "month"
        elif len(date_str) == 4:
            return datetime.strptime(date_str, "%Y"), "year"
        else:
            return None, "invalid"
    except ValueError:
        return None, "invalid"


def check_ae_dates_v2(ae_df, dm_df):
    merged = ae_df.merge(dm_df[["USUBJID", "RFSTDTC"]], on="USUBJID", how="left")
    issues = []

    for _, row in merged.iterrows():
        ae_date, ae_prec = parse_partial_date(row.get("AESTDTC"))
        ref_date, ref_prec = parse_partial_date(row.get("RFSTDTC"))

        if ae_date is None or ref_date is None:
            continue  # missing/invalid dates — a separate check, not this one

        if ae_date < ref_date:
            if ae_prec == "day" and ref_prec == "day":
                # Both fully precise — a real, confirmed comparison
                days_before = (ref_date - ae_date).days
                issues.append({
                    "domain": "AE",
                    "subject": row["USUBJID"],
                    "category": "confirmed_before_reference",
                    "issue": f"AE started {days_before} day(s) before reference start "
                             f"({row['AESTDTC']} vs {row['RFSTDTC']}) — likely screening-period AE"
                })
            else:
                # Imprecise date on one or both sides — can't confirm, needs human review
                issues.append({
                    "domain": "AE",
                    "subject": row["USUBJID"],
                    "category": "imprecise_date_review_needed",
                    "issue": f"AE date ({row['AESTDTC']}, precision={ae_prec}) appears before "
                             f"reference start ({row['RFSTDTC']}, precision={ref_prec}) — "
                             f"comparison uncertain due to incomplete date"
                })

    return issues


ae_date_issues_v2 = check_ae_dates_v2(ae, dm)
report_v2 = pd.DataFrame(ae_date_issues_v2)

print(f"Total flagged: {len(report_v2)}")
print(report_v2["category"].value_counts())
report_v2.head(20)

Total flagged: 65
category
confirmed_before_reference      45
imprecise_date_review_needed    20
Name: count, dtype: int64


,domain,subject,category,issue
0,AE,01-701-1111,confirmed_before_reference,AE started 5 day(s) before reference start (20...
1,AE,01-701-1111,confirmed_before_reference,AE started 5 day(s) before reference start (20...
2,AE,01-701-1111,confirmed_before_reference,AE started 61 day(s) before reference start (2...
3,AE,01-701-1111,confirmed_before_reference,AE started 5 day(s) before reference start (20...
4,AE,01-701-1111,confirmed_before_reference,AE started 5 day(s) before reference start (20...
5,AE,01-701-1118,imprecise_date_review_needed,"AE date (2003, precision=year) appears before ..."
6,AE,01-701-1146,confirmed_before_reference,AE started 4 day(s) before reference start (20...
7,AE,01-701-1146,confirmed_before_reference,AE started 4 day(s) before reference start (20...
8,AE,01-701-1148,confirmed_before_reference,AE started 25 day(s) before reference start (2...
9,AE,01-701-1148,imprecise_date_review_needed,"AE date (2012-02, precision=month) appears bef..."


**Generate report regarding QC issues**

This final step combines the results of all four checks into one report and optionally you can save it into a file.

In [24]:
save_to_file = True  # Set this to False if you do not want to save a CSV file

all_issues = dm_issues + vs_issues + dm_dupes + ae_date_issues_v2
report = pd.DataFrame(all_issues)
report = report[["domain", "subject", "category", "issue"]]

if save_to_file:
    report.to_csv("clinical_data_qc_report.csv", index=False)
    print("Report saved to clinical_data_qc_report.csv")

print(f"Total issues flagged: {len(report)}")
print("\nBreakdown by category:")
print(report["category"].value_counts())
print("\nBreakdown by domain:")
print(report["domain"].value_counts())

report

Report saved to clinical_data_qc_report.csv
Total issues flagged: 72

Breakdown by category:
category
confirmed_before_reference      45
imprecise_date_review_needed    20
out_of_range_value               7
Name: count, dtype: int64

Breakdown by domain:
domain
AE    65
VS     7
Name: count, dtype: int64


,domain,subject,category,issue
0,VS,01-701-1203,out_of_range_value,DIABP value 39.0 outside expected range (40-120)
1,VS,01-701-1203,out_of_range_value,DIABP value 39.0 outside expected range (40-120)
2,VS,01-701-1345,out_of_range_value,DIABP value 39.0 outside expected range (40-120)
3,VS,01-706-1384,out_of_range_value,SYSBP value 217.0 outside expected range (70-200)
4,VS,01-708-1158,out_of_range_value,SYSBP value 208.0 outside expected range (70-200)
...,...,...,...,...
67,AE,01-717-1004,confirmed_before_reference,AE started 1 day(s) before reference start (20...
68,AE,01-717-1004,confirmed_before_reference,AE started 1 day(s) before reference start (20...
69,AE,01-717-1004,imprecise_date_review_needed,"AE date (2013-05, precision=month) appears bef..."
70,AE,01-717-1357,imprecise_date_review_needed,"AE date (1994-04, precision=month) appears bef..."
